In [1]:
import os
import sys
import numpy as np
from torch.utils.data import Dataset
import torch
from finetune_model_dunwei import ft_1lead_ECGFounder
import pandas as pd
from tqdm import tqdm
import gc

In [2]:
base_data_dir = r'D:\M143020071\MACE and MI\raw_data_result\iSKNA_signal\ch1\sr500_0.5_50_MI_win10s_step2s_2-7m'
base_save_dir = r'D:\M143020071\MACE and MI\xgboost_results\ECG_Founder_feature'
categories = ['MACE', 'MI', 'non_MACE', 'non_MI']
if not os.path.exists(base_save_dir):
    os.makedirs(base_save_dir)

In [3]:
gpu_id = 0
device = torch.device('cuda:{}'.format(gpu_id) if torch.cuda.is_available() else 'cpu')
n_classes = 150
pth = r'F:\M143020071\MI\program\ECG_Founder\checkpoint\1_lead_ECGFounder.pth'
model = ft_1lead_ECGFounder(device, pth, n_classes, linear_prob=False)
model.eval()

# 3. 讀取所有資料夾內的資料並「一口氣」合併
all_signals = []
all_labels = []
all_IDs = []

In [4]:
batch_size = 1024
for cat in categories:
    print(f"--- 處理類別: {cat} ---")
    cat_path = os.path.join(base_data_dir, cat)
    
    # 1. 讀取單一類別資料
    cat_signals = []
    cat_labels = []
    cat_IDs = []
    
    files = [f for f in os.listdir(cat_path) if f.endswith('.npy')]
    for filename in tqdm(files, desc=f"Loading {cat}"):
        file_path = os.path.join(cat_path, filename)
        data = np.load(file_path)
        cat_signals.append(data[:, 1:])
        cat_labels.append(data[:, 0])
        cat_IDs.extend([filename.split('.')[0]] * len(data))
    
    signals = np.concatenate(cat_signals, axis=0)
    labels = np.concatenate(cat_labels, axis=0).reshape(-1, 1)
    IDs = np.array(cat_IDs).reshape(-1, 1)
    
    # 2. 立即提取特徵
    feature_list = []
    deep_feature_list = []
    with torch.no_grad():
        for i in range(0, signals.shape[0], batch_size):
            batch_signals = signals[i : i + batch_size]
            batch_tensor = torch.tensor(batch_signals, dtype=torch.float32).to(device).unsqueeze(1)
            
            feats, deep_feats = model(batch_tensor)
            feature_list.append(feats.cpu().numpy())
            deep_feature_list.append(deep_feats.cpu().numpy())
            
    # 3. 合併並儲存該類別的結果
    final_feats = np.concatenate(feature_list, axis=0)
    out_152 = np.concatenate((IDs, labels, final_feats), axis=1)
    np.save(os.path.join(base_save_dir, f'features_{cat}_152.npy'), out_152)

    final_deep_feats = np.concatenate(deep_feature_list, axis=0)
    out_1026 = np.concatenate((IDs, labels, final_deep_feats), axis=1)
    np.save(os.path.join(base_save_dir, f'features_{cat}_1026.npy'), out_1026)
    
    print(f"   - 152維特徵 Shape: {out_152.shape}")
    print(f"   - 1026維特徵 Shape: {out_1026.shape}")

    # 4. 重要：徹底清理記憶體，再進入下一個類別
    del signals, labels, IDs, cat_signals, cat_labels, cat_IDs, feature_list, out_152,deep_feature_list, out_1026
    torch.cuda.empty_cache()
    gc.collect()

print("所有類別特徵提取完成！")

--- 處理類別: MACE ---


Loading MACE: 100%|██████████| 90/90 [00:00<00:00, 699.19it/s]


   - 152維特徵 Shape: (13140, 152)
   - 1026維特徵 Shape: (13140, 1026)
--- 處理類別: MI ---


Loading MI: 100%|██████████| 200/200 [00:06<00:00, 30.78it/s]


   - 152維特徵 Shape: (29200, 152)
   - 1026維特徵 Shape: (29200, 1026)
--- 處理類別: non_MACE ---


Loading non_MACE: 100%|██████████| 432/432 [00:00<00:00, 767.36it/s]


   - 152維特徵 Shape: (63072, 152)
   - 1026維特徵 Shape: (63072, 1026)
--- 處理類別: non_MI ---


Loading non_MI: 100%|██████████| 200/200 [00:06<00:00, 31.32it/s]


   - 152維特徵 Shape: (29200, 152)
   - 1026維特徵 Shape: (29200, 1026)
所有類別特徵提取完成！
